# Phase 1.2: Data Cleaning and Standardization

This notebook processes the raw CSV files downloaded in Phase 1.1. 
Goals:
1. Load all 7 datasets.
2. Standardize coordinate columns (Latitude/Longitude).
3. Standardize Borough names across all datasets.
4. Save cleaned versions to `data/processed/`.

In [4]:
import pandas as pd
import os
import re

# Paths
RAW_DIR = '../data/raw/'
PROCESSED_DIR = '../data/processed/'

if not os.path.exists(PROCESSED_DIR):
    os.makedirs(PROCESSED_DIR)

def extract_lat_long(geom_str):
    """Extracts lat/long from POINT (long lat) strings."""
    if pd.isna(geom_str) or not isinstance(geom_str, str):
        return None, None
    match = re.search(r'POINT \(([-\d\.]+) ([-\d\.]+)\)', geom_str)
    if match:
        return float(match.group(2)), float(match.group(1))
    return None, None

# Borough standardization mapping
BORO_MAP = {
    'M': 'Manhattan', 'B': 'Brooklyn', 'X': 'Bronx', 'Q': 'Queens', 'R': 'Staten Island',
    'BK': 'Brooklyn', 'BX': 'Bronx', 'QN': 'Queens', 'SI': 'Staten Island', 'MN': 'Manhattan',
    'RICHMOND': 'Staten Island'
}

def standardize_boro(boro):
    if pd.isna(boro) or not isinstance(boro, str):
        return boro
    boro = boro.strip()
    title_boro = boro.title()
    if title_boro in ['Manhattan', 'Brooklyn', 'Bronx', 'Queens', 'Staten Island']:
        return title_boro
    return BORO_MAP.get(boro.upper(), title_boro)

print("Setup complete.")

Setup complete.


## 1. Urban Resource Datasets (Spatial)

In [ ]:
def clean_spatial_dataset(filename, boro_col, lat_col=None, lon_col=None, geom_col=None, cols_to_keep=None):
    print(f"Cleaning {filename}...")
    df = pd.read_csv(os.path.join(RAW_DIR, filename))
    
    if geom_col:
        df['latitude'], df['longitude'] = zip(*df[geom_col].apply(extract_lat_long))
    elif lat_col and lon_col:
        df['latitude'] = pd.to_numeric(df[lat_col], errors='coerce')
        df['longitude'] = pd.to_numeric(df[lon_col], errors='coerce')
    
    df[boro_col] = df[boro_col].apply(standardize_boro)
    
    if cols_to_keep:
        df = df[cols_to_keep + ['latitude', 'longitude', boro_col]]
    
    # 1. DROP MISSING VALUES
    df.dropna(subset=['latitude', 'longitude'], inplace=True)
    
    # 2. SPATIAL FILTER (THE FIX): Remove (0,0) and extreme outliers
    # NYC is roughly Latitude 40.5-40.9 and Longitude -74.3 to -73.7
    df = df[
        (df['latitude'] > 40.0) & (df['latitude'] < 42.0) &
        (df['longitude'] > -75.0) & (df['longitude'] < -72.0)
    ]
    
    output_name = filename.replace('.csv', '_cleaned.csv')
    df.to_csv(os.path.join(PROCESSED_DIR, output_name), index=False)
    return len(df)


counts = {}
counts['fountains'] = clean_spatial_dataset('drinking_fountains.csv', 'borough', geom_col='the_geom', cols_to_keep=['propertyna'])
counts['toilets'] = clean_spatial_dataset('public_toilets.csv', 'boro_name', geom_col='the_geom', cols_to_keep=['site_name'])
counts['centers'] = clean_spatial_dataset('drop_in_centers.csv', 'borough', lat_col='latitude', lon_col='longitude', cols_to_keep=['center_name'])
counts['link_nyc'] = clean_spatial_dataset('link_nyc.csv', 'boro', lat_col='latitude', lon_col='longitude', cols_to_keep=['neighborhood_tabulation_area_nta'])
counts['drug_crime'] = clean_spatial_dataset('drug_crime.csv', 'boro_nm', lat_col='latitude', lon_col='longitude', cols_to_keep=['cmplnt_fr_dt', 'ofns_desc'])

for k, v in counts.items():
    print(f"{k}: {v} rows")

Cleaning drinking_fountains.csv...
Cleaning public_toilets.csv...
Cleaning drop_in_centers.csv...
Cleaning link_nyc.csv...
Cleaning drug_crime.csv...
fountains: 3849 rows
toilets: 7 rows
centers: 8 rows
link_nyc: 2255 rows
drug_crime: 490666 rows


## 2. Non-Spatial & Complex Datasets (Shelters & Census)

In [6]:
print("Cleaning shelter_repair and rhy_census...")
df_shelter = pd.read_csv(os.path.join(RAW_DIR, 'shelter_repair.csv'))
df_shelter.to_csv(os.path.join(PROCESSED_DIR, 'shelter_repair_cleaned.csv'), index=False)

df_rhy = pd.read_csv(os.path.join(RAW_DIR, 'rhy_census.csv'))
df_rhy.to_csv(os.path.join(PROCESSED_DIR, 'rhy_census_cleaned.csv'), index=False)

print("Cleaning complete.")

Cleaning shelter_repair and rhy_census...
Cleaning complete.
